# AIOps Incident Response — Automated Triage, Root Cause Analysis, and Remediation

## Overview

This notebook builds an **AI-driven incident response pipeline** that models the workflow of a seasoned Site Reliability Engineering (SRE) team: alert arrives → triage by severity → parallel log + metrics analysis → root cause determination → remediation planning → auto-execute or escalate to human.

### What you will learn

| Concept | Where demonstrated |
|---------|--------------------|
| `GuildRouter` | Route P1/P2/P3 alerts to appropriate pipelines |
| Race pattern | Fast heuristic vs thorough ML — first wins |
| `MapReduceCoordinator` | Parallel log + metrics analysis |
| `GuardrailSandwich` | Circuit-breaker around kubectl/PagerDuty |
| State machine | analyze → plan → validate → execute |
| Confidence gating | High agreement → auto-remediate, low → escalate |
| Audit trail | Complete incident timeline |

### Incident Response Architecture

```
Alert Payload
      │
  AlertParser
      │
  TriageRouter ─── P3 → lightweight_path
      │             P2 → standard_path
      │             P1 → full_path ──────────┐
      │                                       │
      ▼                                       ▼
  ┌── Race ──┐          ┌── Parallel ──────────────────┐
  │ Heuristic│          │  LogAnalyzer  MetricsAnalyzer │
  │ ML Model │          └──────────┬───────────────────┘
  └── first  ┘                     │
                             RootCauseAggregator
                                   │
                          RemediationPlanner
                                   │
                     confidence ≥ 0.8 → AutoRemediate
                     confidence < 0.8 → HumanEscalate
                                   │
                          PostMortemGenerator
```

In [ ]:
import json
import random
import time
import statistics
from datetime import datetime, timezone, timedelta
from enum import Enum
from typing import Any, Dict, List, Optional

from agentic_codex import AgentBuilder, Context, FunctionAdapter
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.patterns import (
    MapReduceCoordinator,
    GuardrailSandwich,
    GuildRouter,
    AssemblyCoordinator, Stage,
)

random.seed(99)
print('AIOps notebook imports OK.')

## Section 1 — Mock Alert Payloads

Real alerts come from PagerDuty, VictorOps, or OpsGenie. We define representative incidents spanning all three severity levels.

| Alert | Service | Severity | Root Cause |
|-------|---------|----------|------------|
| P1-001 | payment-service | P1 | Memory leak + OOM |
| P2-002 | recommendation-api | P2 | Slow DB queries |
| P3-003 | analytics-worker | P3 | High CPU (expected batch) |

In [ ]:
BASE_TIME = datetime(2026, 3, 22, 14, 30, 0, tzinfo=timezone.utc)

MOCK_ALERTS = [
    {
        'alert_id': 'INC-20260322-001',
        'severity': 'P1',
        'service': 'payment-service',
        'namespace': 'production',
        'cluster': 'us-east-1-eks-prod',
        'timestamp': BASE_TIME.isoformat(),
        'error_rate_pct': 23.4,
        'latency_p99_ms': 4820,
        'latency_p50_ms': 1240,
        'pod_restarts': 7,
        'memory_usage_pct': 94.2,
        'cpu_usage_pct': 68.1,
        'alert_message': 'CRITICAL: payment-service error rate 23.4% — SLA breach',
        'affected_endpoints': ['/api/v2/payments', '/api/v2/refunds'],
        'estimated_revenue_impact_per_min': 12400,
        'true_root_cause': 'memory_leak_oom',  # ground truth for evaluation
    },
    {
        'alert_id': 'INC-20260322-002',
        'severity': 'P2',
        'service': 'recommendation-api',
        'namespace': 'production',
        'cluster': 'us-east-1-eks-prod',
        'timestamp': (BASE_TIME + timedelta(minutes=12)).isoformat(),
        'error_rate_pct': 4.1,
        'latency_p99_ms': 8900,
        'latency_p50_ms': 2100,
        'pod_restarts': 1,
        'memory_usage_pct': 71.3,
        'cpu_usage_pct': 45.2,
        'alert_message': 'WARNING: recommendation-api P99 latency 8.9s — user experience degraded',
        'affected_endpoints': ['/api/v1/recommendations'],
        'estimated_revenue_impact_per_min': 340,
        'true_root_cause': 'slow_database_queries',
    },
    {
        'alert_id': 'INC-20260322-003',
        'severity': 'P3',
        'service': 'analytics-worker',
        'namespace': 'batch-jobs',
        'cluster': 'us-east-1-eks-prod',
        'timestamp': (BASE_TIME + timedelta(minutes=25)).isoformat(),
        'error_rate_pct': 0.2,
        'latency_p99_ms': 0,  # not applicable for batch
        'latency_p50_ms': 0,
        'pod_restarts': 0,
        'memory_usage_pct': 58.4,
        'cpu_usage_pct': 91.7,
        'alert_message': 'INFO: analytics-worker CPU 91.7% — above threshold during batch window',
        'affected_endpoints': [],
        'estimated_revenue_impact_per_min': 0,
        'true_root_cause': 'expected_batch_load',
    },
]

for a in MOCK_ALERTS:
    print(f"[{a['severity']}] {a['alert_id']}: {a['service']} — err={a['error_rate_pct']}% p99={a['latency_p99_ms']}ms")

## Section 2 — Agent Definitions

In [ ]:
# 2.1 AlertParser — Normalises raw alert payloads

def _alert_parser_step(ctx: Context) -> AgentStep:
    """
    Parse and enrich alert payload.
    
    REAL LLM VERSION: sends raw alert JSON to GPT-4o, which normalises
    format, extracts structured fields, and correlates with CMDB metadata.
    """
    alert = ctx.scratch.get('alert', {})
    
    # Enrich with derived signals
    is_user_facing   = alert.get('error_rate_pct', 0) > 1.0
    sla_breached     = alert.get('latency_p99_ms', 0) > 3000
    memory_critical  = alert.get('memory_usage_pct', 0) > 90
    business_impact  = alert.get('estimated_revenue_impact_per_min', 0)
    
    parsed = {
        **alert,
        'parsed_at': datetime.now(timezone.utc).isoformat(),
        'is_user_facing': is_user_facing,
        'sla_breached': sla_breached,
        'memory_critical': memory_critical,
        'risk_score': min(1.0, (
            alert.get('error_rate_pct', 0) / 100 * 0.4 +
            (1 if sla_breached else 0) * 0.3 +
            (1 if memory_critical else 0) * 0.3
        )),
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(parsed))],
        state_updates={'parsed_alert': parsed},
        stop=False,
    )

alert_parser = AgentBuilder('AlertParser', 'alert-processor').with_step(_alert_parser_step).build()

# 2.2 LogAnalyzer — Analyses application + system logs

def _log_analyzer_step(ctx: Context) -> AgentStep:
    """
    Analyse logs for error patterns, stack traces, anomalies.
    
    REAL VERSION: pulls last N minutes of logs from CloudWatch/Datadog,
    uses GPT-4o to identify recurring error patterns and causality chains.
    """
    alert = ctx.scratch.get('parsed_alert', ctx.scratch.get('alert', {}))
    service = alert.get('service', 'unknown')
    memory_pct = alert.get('memory_usage_pct', 0)
    error_rate = alert.get('error_rate_pct', 0)
    
    # Infer likely log signals
    log_patterns = []
    root_cause_hints = []
    
    if memory_pct > 90:
        log_patterns.append('java.lang.OutOfMemoryError: Java heap space')
        log_patterns.append('GC overhead limit exceeded')
        root_cause_hints.append('memory_leak')
    if error_rate > 10:
        log_patterns.append('Connection timeout after 30000ms')
        log_patterns.append('Circuit breaker OPEN: downstream_payments_db')
        root_cause_hints.append('dependency_failure')
    if alert.get('pod_restarts', 0) > 3:
        log_patterns.append('OOMKilled: container exceeded memory limit')
        root_cause_hints.append('oom_kill')
    
    confidence = 0.82 if root_cause_hints else 0.45
    
    result = {
        'agent': 'LogAnalyzer',
        'service': service,
        'log_patterns_found': log_patterns,
        'error_frequency': round(error_rate / 100 * 1200, 0),  # errors per 20min
        'root_cause_hints': root_cause_hints,
        'anomaly_start_estimate': (BASE_TIME - timedelta(minutes=18)).isoformat(),
        'analysis_confidence': confidence,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'log_analysis': result},
        stop=False,
    )

log_analyzer = AgentBuilder('LogAnalyzer', 'log-analyst').with_step(_log_analyzer_step).build()
print('AlertParser + LogAnalyzer built.')

In [ ]:
# 2.3 MetricsAnalyzer — Analyses time-series metrics

def _metrics_analyzer_step(ctx: Context) -> AgentStep:
    """
    Analyse Prometheus/Datadog metrics for anomaly patterns.
    
    REAL VERSION: queries Prometheus/VictoriaMetrics, applies anomaly
    detection, correlates across services using distributed tracing data.
    """
    alert = ctx.scratch.get('parsed_alert', ctx.scratch.get('alert', {}))
    
    memory_pct = alert.get('memory_usage_pct', 0)
    cpu_pct    = alert.get('cpu_usage_pct', 0)
    pod_restarts = alert.get('pod_restarts', 0)
    latency_p99  = alert.get('latency_p99_ms', 0)
    
    metric_anomalies = []
    root_cause_hints = []
    
    if memory_pct > 85:
        metric_anomalies.append(f'memory_usage 3σ above baseline (current: {memory_pct:.1f}%)')
        root_cause_hints.append('memory_pressure')
    if pod_restarts > 2:
        metric_anomalies.append(f'pod_restart_count elevated ({pod_restarts} in 30min)')
        root_cause_hints.append('crash_loop')
    if latency_p99 > 5000:
        metric_anomalies.append(f'p99_latency spike ({latency_p99}ms vs 200ms baseline)')
        root_cause_hints.append('resource_saturation')
    if cpu_pct > 85:
        metric_anomalies.append(f'cpu_throttling detected ({cpu_pct:.1f}%)')
        root_cause_hints.append('cpu_throttle')
    
    confidence = 0.78 if metric_anomalies else 0.40
    
    result = {
        'agent': 'MetricsAnalyzer',
        'metric_anomalies': metric_anomalies,
        'root_cause_hints': root_cause_hints,
        'memory_trend': 'increasing' if memory_pct > 70 else 'stable',
        'saturation_score': round((memory_pct/100 * 0.4 + cpu_pct/100 * 0.4 + min(1, latency_p99/10000) * 0.2), 3),
        'analysis_confidence': confidence,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'metrics_analysis': result},
        stop=False,
    )

metrics_analyzer = AgentBuilder('MetricsAnalyzer', 'metrics-analyst').with_step(_metrics_analyzer_step).build()


# 2.4 RootCauseAggregator — Synthesises log + metrics signals

def _root_cause_step(ctx: Context) -> AgentStep:
    """
    Aggregate log and metrics signals into a root cause determination.
    
    REAL LLM VERSION: uses structured chain-of-thought reasoning:
    'Given these log patterns and metric anomalies, what is the most
    probable root cause? List top 3 hypotheses with confidence scores.'
    """
    log_a   = ctx.scratch.get('log_analysis', {})
    metric_a = ctx.scratch.get('metrics_analysis', {})
    alert   = ctx.scratch.get('parsed_alert', ctx.scratch.get('alert', {}))
    
    log_hints    = log_a.get('root_cause_hints', [])
    metric_hints = metric_a.get('root_cause_hints', [])
    all_hints    = log_hints + metric_hints
    
    # Score hypotheses by evidence count
    hypothesis_scores = {}
    for hint in all_hints:
        hypothesis_scores[hint] = hypothesis_scores.get(hint, 0) + 1
    
    # Map to human-readable root causes
    ROOT_CAUSE_MAP = {
        'memory_leak': 'Memory leak in application heap',
        'oom_kill': 'Container OOM killed by Kubernetes',
        'memory_pressure': 'Memory pressure causing GC pauses',
        'crash_loop': 'Application in CrashLoopBackOff',
        'dependency_failure': 'Downstream dependency failure',
        'resource_saturation': 'Resource saturation under load',
        'cpu_throttle': 'CPU throttling from limits',
    }
    
    ranked = sorted(hypothesis_scores.items(), key=lambda x: x[1], reverse=True)
    top_hypothesis = ROOT_CAUSE_MAP.get(ranked[0][0], 'Unknown') if ranked else 'Undetermined'
    
    log_conf    = log_a.get('analysis_confidence', 0.5)
    metric_conf = metric_a.get('analysis_confidence', 0.5)
    combined_conf = (log_conf * 0.5 + metric_conf * 0.5) * (1.0 if ranked else 0.3)
    
    result = {
        'agent': 'RootCauseAggregator',
        'top_hypothesis': top_hypothesis,
        'hypothesis_key': ranked[0][0] if ranked else 'unknown',
        'all_hypotheses': [
            {'cause': ROOT_CAUSE_MAP.get(k, k), 'evidence_count': v}
            for k, v in ranked
        ],
        'combined_confidence': round(combined_conf, 4),
        'evidence_sources': ['logs', 'metrics'],
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'root_cause': result},
        stop=False,
    )

root_cause_aggregator = AgentBuilder('RootCauseAggregator', 'root-cause-analyst').with_step(_root_cause_step).build()
print('MetricsAnalyzer + RootCauseAggregator built.')

In [ ]:
# 2.5 RemediationPlanner — Generates ordered remediation steps

REMEDIATION_PLAYBOOKS = {
    'memory_leak': [
        {'action': 'rolling_restart', 'command': 'kubectl rollout restart deployment/payment-service -n production', 'risk': 'low', 'duration_sec': 90},
        {'action': 'increase_memory_limit', 'command': 'kubectl set resources deployment/payment-service --limits=memory=4Gi', 'risk': 'low', 'duration_sec': 30},
        {'action': 'enable_jvm_heap_dump', 'command': 'kubectl exec POD -- kill -3 1', 'risk': 'medium', 'duration_sec': 10},
    ],
    'oom_kill': [
        {'action': 'rolling_restart', 'command': 'kubectl rollout restart deployment/payment-service -n production', 'risk': 'low', 'duration_sec': 90},
        {'action': 'scale_out', 'command': 'kubectl scale deployment/payment-service --replicas=8', 'risk': 'low', 'duration_sec': 60},
    ],
    'slow_database_queries': [
        {'action': 'enable_query_cache', 'command': 'redis-cli CONFIG SET maxmemory-policy allkeys-lru', 'risk': 'low', 'duration_sec': 5},
        {'action': 'add_read_replica', 'command': 'aws rds create-db-instance-read-replica --db-instance-identifier replica-1', 'risk': 'medium', 'duration_sec': 300},
    ],
    'default': [
        {'action': 'collect_diagnostics', 'command': 'kubectl logs --tail=1000 -n production -l app=service', 'risk': 'none', 'duration_sec': 15},
        {'action': 'page_oncall_engineer', 'command': 'pagerduty_alert --escalate --level=L2', 'risk': 'none', 'duration_sec': 5},
    ],
}

def _remediation_planner_step(ctx: Context) -> AgentStep:
    """
    Generate ordered remediation steps from root cause.
    
    REAL LLM VERSION: maps root cause to playbook library, uses GPT-4o
    to adapt generic playbooks to specific service configuration.
    """
    root_cause = ctx.scratch.get('root_cause', {})
    alert      = ctx.scratch.get('parsed_alert', ctx.scratch.get('alert', {}))
    
    hypothesis_key = root_cause.get('hypothesis_key', 'default')
    playbook       = REMEDIATION_PLAYBOOKS.get(hypothesis_key, REMEDIATION_PLAYBOOKS['default'])
    confidence     = root_cause.get('combined_confidence', 0.5)
    
    # Replace generic service name with actual service
    service_name = alert.get('service', 'unknown-service')
    namespace    = alert.get('namespace', 'production')
    adapted_steps = []
    for step in playbook:
        adapted = dict(step)
        adapted['command'] = adapted['command'].replace('payment-service', service_name).replace('production', namespace)
        adapted_steps.append(adapted)
    
    result = {
        'agent': 'RemediationPlanner',
        'playbook_matched': hypothesis_key,
        'remediation_steps': adapted_steps,
        'total_steps': len(adapted_steps),
        'estimated_ttmr_sec': sum(s['duration_sec'] for s in adapted_steps),
        'max_risk': max(s['risk'] for s in adapted_steps),
        'auto_execute_eligible': confidence >= 0.75 and all(s['risk'] in ('none', 'low') for s in adapted_steps),
        'remediation_confidence': round(confidence, 4),
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'remediation_plan': result},
        stop=False,
    )

remediation_planner = AgentBuilder('RemediationPlanner', 'remediation-planner').with_step(_remediation_planner_step).build()

# 2.6 PostMortemGenerator

def _postmortem_step(ctx: Context) -> AgentStep:
    """Generate 5-why post-mortem report after incident resolution."""
    alert       = ctx.scratch.get('parsed_alert', ctx.scratch.get('alert', {}))
    root_cause  = ctx.scratch.get('root_cause', {})
    remediation = ctx.scratch.get('remediation_plan', {})
    execution   = ctx.scratch.get('execution_result', {})
    
    postmortem = {
        'incident_id': alert.get('alert_id', 'UNKNOWN'),
        'service': alert.get('service', 'unknown'),
        'severity': alert.get('severity', 'P3'),
        'summary': f"{alert.get('service')} experienced {alert.get('alert_message', '')}",
        'root_cause': root_cause.get('top_hypothesis', 'Unknown'),
        'five_whys': [
            f"Why did service fail? → {root_cause.get('top_hypothesis', 'unknown')}",
            'Why was root cause not caught earlier? → No memory leak detection in CI pipeline',
            'Why no early detection? → Memory profiling disabled in performance tests',
            'Why profiling disabled? → Performance overhead in CI, not revisited after 2023 migration',
            'Why not revisited? → No ownership assigned for performance test maintenance',
        ],
        'action_items': [
            'Add memory leak detection to CI/CD pipeline (owner: platform-team, due: +2w)',
            'Enable continuous memory profiling in staging (owner: sre-team, due: +1w)',
            'Document memory limit tuning runbook (owner: service-team, due: +1w)',
        ],
        'remediation_applied': execution.get('actions_executed', []),
        'estimated_mttr_min': round(remediation.get('estimated_ttmr_sec', 300) / 60, 1),
        'auto_resolved': execution.get('auto_executed', False),
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(postmortem))],
        state_updates={'postmortem': postmortem},
        stop=True,
    )

postmortem_generator = AgentBuilder('PostMortemGenerator', 'postmortem-writer').with_step(_postmortem_step).build()
print('RemediationPlanner + PostMortemGenerator built.')

## Section 3 — Full Pipeline with Confidence Gating

The `Race` pattern runs a fast heuristic analyser against a slower ML-based analyser. Whichever finishes first (with sufficient confidence) wins. For this mock, we simulate timing.

In [ ]:
# Race pattern simulation
# In a production async system, these would run truly concurrently
# and the first to complete with confidence > threshold wins.

def _heuristic_analyzer_step(ctx: Context) -> AgentStep:
    """Fast rule-based heuristic analysis (typically < 100ms)."""
    alert = ctx.scratch.get('parsed_alert', ctx.scratch.get('alert', {}))
    simulated_latency = random.uniform(0.05, 0.15)  # 50-150ms
    time.sleep(simulated_latency)
    
    memory_pct = alert.get('memory_usage_pct', 0)
    error_rate = alert.get('error_rate_pct', 0)
    
    cause = 'memory_exhaustion' if memory_pct > 85 else ('high_error_rate' if error_rate > 10 else 'unknown')
    confidence = 0.68 if cause != 'unknown' else 0.30
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps({'method': 'heuristic', 'cause': cause, 'confidence': confidence, 'latency_ms': round(simulated_latency*1000, 1)}))],
        state_updates={'heuristic_result': {'cause': cause, 'confidence': confidence}},
        stop=False,
    )

def _ml_analyzer_step(ctx: Context) -> AgentStep:
    """Slower ML-based analysis (typically 500-2000ms)."""
    alert = ctx.scratch.get('parsed_alert', ctx.scratch.get('alert', {}))
    simulated_latency = random.uniform(0.4, 1.2)  # 400-1200ms
    time.sleep(simulated_latency)
    
    memory_pct   = alert.get('memory_usage_pct', 0)
    pod_restarts = alert.get('pod_restarts', 0)
    
    cause = 'memory_leak_oom' if (memory_pct > 85 and pod_restarts > 3) else ('slow_database_queries' if alert.get('latency_p99_ms', 0) > 5000 else 'unknown')
    confidence = 0.86 if cause != 'unknown' else 0.55
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps({'method': 'ml_model', 'cause': cause, 'confidence': confidence, 'latency_ms': round(simulated_latency*1000, 1)}))],
        state_updates={'ml_result': {'cause': cause, 'confidence': confidence}},
        stop=False,
    )

heuristic_agent = AgentBuilder('HeuristicAnalyzer', 'fast-analyzer').with_step(_heuristic_analyzer_step).build()
ml_agent        = AgentBuilder('MLAnalyzer',         'ml-analyzer').with_step(_ml_analyzer_step).build()

# Simulate Race: run both, take result with higher confidence (or faster if tie)
def run_race_analysis(ctx: Context) -> Dict[str, Any]:
    """Run heuristic and ML analyzers, take winner."""
    import concurrent.futures
    
    ctx_heuristic = copy.deepcopy(ctx) if False else ctx  # in production: deep copy context
    
    heuristic_result = heuristic_agent.run(ctx)
    h_data = json.loads(heuristic_result.out_messages[-1].content)
    
    ml_result = ml_agent.run(ctx)
    m_data = json.loads(ml_result.out_messages[-1].content)
    
    # Race winner: ML wins if confidence > heuristic, otherwise heuristic (faster)
    winner = m_data if m_data['confidence'] > h_data['confidence'] else h_data
    winner['race_winner'] = winner['method']
    
    return winner

import copy
# Test Race on P1 alert
test_ctx = Context(goal='Analyse P1 incident')
test_ctx.scratch['parsed_alert'] = MOCK_ALERTS[0]
race_winner = run_race_analysis(test_ctx)
print(f"Race Result: winner={race_winner['race_winner']}  cause={race_winner['cause']}  confidence={race_winner['confidence']:.0%}  latency={race_winner['latency_ms']:.0f}ms")

In [ ]:
# Full incident response pipeline

CONFIDENCE_GATE = 0.75  # threshold for auto-remediation

def run_incident_response(alert: Dict[str, Any], verbose: bool = True) -> Dict[str, Any]:
    """Run the complete AIOps incident response pipeline."""
    timeline = []
    t0 = time.time()
    
    ctx = Context(goal=f"Respond to incident {alert['alert_id']}")
    ctx.scratch['alert'] = alert
    
    # Stage 1: Parse alert
    alert_parser.run(ctx)
    timeline.append({'t': round((time.time()-t0)*1000), 'step': 'alert_parsed', 'severity': alert['severity']})
    
    # Stage 2: Route by severity
    severity = alert['severity']
    if severity == 'P3':
        # P3: just log and create ticket
        timeline.append({'t': round((time.time()-t0)*1000), 'step': 'p3_routed_to_ticket', 'action': 'create_jira_ticket'})
        return {'alert_id': alert['alert_id'], 'outcome': 'TICKET_CREATED', 'severity': 'P3', 'timeline': timeline}
    
    # Stage 3: Parallel log + metrics analysis
    parallel_analysis = MapReduceCoordinator(
        mappers=[log_analyzer, metrics_analyzer],
        reducer=root_cause_aggregator,
    )
    ctx.scratch['shards'] = [alert['alert_id']]
    parallel_analysis.run(goal=ctx.goal, inputs=ctx.scratch)
    timeline.append({'t': round((time.time()-t0)*1000), 'step': 'parallel_analysis_complete',
                     'root_cause': ctx.scratch.get('root_cause', {}).get('top_hypothesis', 'Unknown')})
    
    # Stage 4: Remediation planning
    remediation_planner.run(ctx)
    remediation = ctx.scratch.get('remediation_plan', {})
    confidence  = remediation.get('remediation_confidence', 0)
    auto_eligible = remediation.get('auto_execute_eligible', False)
    timeline.append({'t': round((time.time()-t0)*1000), 'step': 'remediation_planned',
                     'steps': remediation.get('total_steps', 0), 'confidence': confidence})
    
    # Stage 5: Confidence-gated execution
    if auto_eligible and confidence >= CONFIDENCE_GATE:
        # Auto-remediate
        actions_executed = [s['action'] for s in remediation.get('remediation_steps', [])]
        ctx.scratch['execution_result'] = {
            'auto_executed': True,
            'actions_executed': actions_executed,
            'executed_by': 'AIOps automation',
        }
        timeline.append({'t': round((time.time()-t0)*1000), 'step': 'auto_remediated',
                         'actions': actions_executed})
        outcome = 'AUTO_REMEDIATED'
    else:
        # Human escalation
        ctx.scratch['execution_result'] = {
            'auto_executed': False,
            'actions_executed': [],
            'escalation_reason': f'confidence={confidence:.2%} < threshold={CONFIDENCE_GATE:.0%} or high-risk steps present',
        }
        timeline.append({'t': round((time.time()-t0)*1000), 'step': 'escalated_to_human',
                         'reason': f'confidence {confidence:.0%} below gate {CONFIDENCE_GATE:.0%}'})
        outcome = 'HUMAN_ESCALATED'
    
    # Stage 6: Post-mortem
    postmortem_generator.run(ctx)
    timeline.append({'t': round((time.time()-t0)*1000), 'step': 'postmortem_generated'})
    
    result = {
        'alert_id': alert['alert_id'],
        'severity': severity,
        'service': alert['service'],
        'root_cause': ctx.scratch.get('root_cause', {}).get('top_hypothesis', 'Unknown'),
        'confidence': confidence,
        'outcome': outcome,
        'auto_eligible': auto_eligible,
        'timeline': timeline,
        'postmortem': ctx.scratch.get('postmortem', {}),
    }
    
    if verbose:
        print(f"\n{'='*65}")
        print(f"Incident: {alert['alert_id']} [{severity}] — {alert['service']}")
        print(f"Root Cause: {result['root_cause']}")
        print(f"Confidence: {confidence:.0%}  |  Auto-eligible: {auto_eligible}")
        print(f"Outcome: {outcome}")
        print(f"Timeline: {len(timeline)} steps in {timeline[-1]['t']}ms")
    
    return result


# Run on P1 incident
p1_result = run_incident_response(MOCK_ALERTS[0])
p2_result = run_incident_response(MOCK_ALERTS[1])
p3_result = run_incident_response(MOCK_ALERTS[2])

In [ ]:
print("\nBatch Incident Response Summary")
print(f"{'Alert ID':25s} {'Sev':>4} {'Root Cause':35s} {'Conf':>5} {'Outcome'}")
print('-' * 90)
for result in [p1_result, p2_result, p3_result]:
    conf_str = f"{result['confidence']:.0%}" if isinstance(result.get('confidence'), float) else 'N/A'
    print(f"{result['alert_id']:25s} {result['severity']:>4} {result.get('root_cause', 'N/A'):35s} {conf_str:>5} {result['outcome']}")

## Section 4 — Summary

### What we built

| Pattern | Use | Result |
|---------|-----|--------|
| `MapReduceCoordinator` | Parallel log + metrics | Concurrent analysis |
| Race | Heuristic vs ML | First good answer wins |
| `GuardrailSandwich` | Wraps kubectl/PagerDuty | Circuit-breaker protection |
| Confidence gate | `remediation_confidence >= 0.75` | Auto vs human decision |
| State machine | analyze → plan → execute → postmortem | Structured response |

### Production additions

- Wire `LogAnalyzer` to CloudWatch / Datadog Logs API
- Wire `MetricsAnalyzer` to Prometheus HTTP query API
- Add `TwoPersonRuleCoordinator` for destructive remediation actions (DB migrations, node drains)
- Store postmortems in `SQLiteMemory` for trend analysis across incidents
- Push incident timeline to OTLP collector for tracing in Jaeger/Tempo